In [3]:
!nvidia-smi

/env/miniconda3/envs/pytorch/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


Tue Feb 10 04:07:34 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.183.01             Driver Version: 535.183.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA RTX A6000               Off | 00000000:01:00.0 Off |                  Off |
| 58%   83C    P2             173W / 300W |  36848MiB / 49140MiB |     99%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [1]:

# ============================================================
# TRADITIONAL ALM-PINN FOR REDUCED COUPLED NEUTRON / THERMAL SYSTEM
#
# Network outputs:
#   [phi, Ts, Tf]
#
# Physics:
#   phi_t - D_phi (phi_xx + phi_yy) + Sigma_a phi = 0          in fuel
#   Ts_t  - k_s   (Ts_xx  + Ts_yy ) - gamma phi = 0             in fuel
#   Tf_t  + v_y Tf_y - k_f (Tf_xx + Tf_yy) = 0                  in coolant
#
# Fixed points, no jitter, no resampling.
#
# Objective:
#   PDE residual MSE on larger fixed objective point sets
#
# ALM constraints:
#   PDE anchors + BC + IC + interface constraints
#
# Traditional ALM:
#   theta inner update: L-BFGS-B for a fixed number of inner iterations
#   lambda update:      lambda <- lambda + rho*c
#   rho update:         blockwise, only after outer iterations
#
# Total training budget:
#   TOTAL_THETA_STEPS = MAX_OUTER * INNER_STEPS
#
# Start with TOTAL_LBFGS_BUDGET = 50000. Then try 50000.
#
# If GPU memory is too high, set USE_X64=False.
# ============================================================

import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2")
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ.setdefault("XLA_FLAGS", "--xla_gpu_autotune_level=0")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import time
import math
from functools import partial
from pathlib import Path
import numpy as np
import scipy.optimize

import jax
jax.config.update("jax_default_matmul_precision", "tensorfloat32")
import jax.numpy as jnp
from jax import random, vmap, jacrev, hessian


# ============================================================
# Precision
# ============================================================
USE_X64 = True
jax.config.update("jax_enable_x64", USE_X64)
DTYPE = jnp.float64 if USE_X64 else jnp.float32
EPS = DTYPE(1e-12)


# ============================================================
# Geometry
# ============================================================
Ls = DTYPE(0.4)
Lf = DTYPE(0.6)
Ly = DTYPE(1.0)
T_end = DTYPE(1.0)

x_min = DTYPE(0.0)
x_max = DTYPE(Ls + Lf)
y_min = DTYPE(0.0)
y_max = DTYPE(Ly)
t_min = DTYPE(0.0)
t_max = DTYPE(T_end)


# ============================================================
# Sample counts: same as the uploaded SQP setup
# ============================================================
# Objective PDE sets
NX_OBJ_PHI, NY_OBJ_PHI, NT_OBJ_PHI = 15, 15, 10
NX_OBJ_TS,  NY_OBJ_TS,  NT_OBJ_TS  = 15, 15, 10
NX_OBJ_TF,  NY_OBJ_TF,  NT_OBJ_TF  = 15, 15, 10

# PDE anchor constraints
NX_CON_PHI, NY_CON_PHI, NT_CON_PHI = 6, 6, 6
NX_CON_TS,  NY_CON_TS,  NT_CON_TS  = 7, 7, 7
NX_CON_TF,  NY_CON_TF,  NT_CON_TF  = 7, 7, 7

K_CON_PHI = NX_CON_PHI * NY_CON_PHI * NT_CON_PHI
K_CON_TS  = NX_CON_TS  * NY_CON_TS  * NT_CON_TS
K_CON_TF  = NX_CON_TF  * NY_CON_TF  * NT_CON_TF

N_PER_CELL = 1
N_BC_PER_BIN = 1
N_IC_PER_BIN = 1
N_IF_PER_BIN = 1

# BC counts
K_BC_PHI_LEFT = 60
K_BC_PHI_Y0   = 60
K_BC_PHI_Y1   = 60
K_BC_PHI_IF   = 60

K_BC_TS_X0    = 60
K_BC_TS_Y0    = 60
K_BC_TS_Y1    = 60

K_BC_TF_Y0    = 60
K_BC_TF_Y1    = 60
K_BC_TF_X1    = 60

# IC counts
NX_IC, NY_IC = 10, 10
K_IC_PHI = NX_IC * NY_IC
K_IC_TS  = NX_IC * NY_IC
K_IC_TF  = NX_IC * NY_IC

# Interface
NY_IF, NT_IF = 10, 10
K_IF = NY_IF * NT_IF

# Block sizes
N_PDE_PHI = K_CON_PHI
N_PDE_TS  = K_CON_TS
N_PDE_TF  = K_CON_TF

N_BC_PHI = K_BC_PHI_LEFT + K_BC_PHI_Y0 + K_BC_PHI_Y1 + K_BC_PHI_IF
N_BC_TS  = K_BC_TS_X0 + K_BC_TS_Y0 + K_BC_TS_Y1
N_BC_TF  = K_BC_TF_Y0 + K_BC_TF_Y1 + K_BC_TF_X1
N_IF     = 2 * K_IF
N_IC     = K_IC_PHI + K_IC_TS + K_IC_TF

M_CON = N_PDE_PHI + N_PDE_TS + N_PDE_TF + N_BC_PHI + N_BC_TS + N_BC_TF + N_IF + N_IC


def make_block_slices():
    s = {}
    i = 0
    s["pde_phi"] = slice(i, i + K_CON_PHI); i += K_CON_PHI
    s["pde_Ts"]  = slice(i, i + K_CON_TS);  i += K_CON_TS
    s["pde_Tf"]  = slice(i, i + K_CON_TF);  i += K_CON_TF
    s["bc_phi"]  = slice(i, i + N_BC_PHI);  i += N_BC_PHI
    s["bc_Ts"]   = slice(i, i + N_BC_TS);   i += N_BC_TS
    s["bc_Tf"]   = slice(i, i + N_BC_TF);   i += N_BC_TF
    s["iface"]   = slice(i, i + N_IF);      i += N_IF
    s["ic"]      = slice(i, i + N_IC);      i += N_IC
    assert i == M_CON, (i, M_CON)
    return s

BLOCKS = make_block_slices()


# ============================================================
# Training knobs
# ============================================================
# L-BFGS-B inner budget. Same ALM structure as the Adam-inner version,
# but each outer loop approximately minimizes the augmented Lagrangian using SciPy L-BFGS-B.
TOTAL_LBFGS_BUDGET = 50000
INNER_MAXITER = 100
MAX_OUTER = int(math.ceil(TOTAL_LBFGS_BUDGET / INNER_MAXITER))

LBFGS_FTOL = 1e-12
LBFGS_GTOL = 1e-8
LBFGS_MAXLS = 50
LBFGS_MAXCOR = 50

VALIDATE_EVERY_OUTER = 10     # set 0 to disable validation during training

# Objective weights: same default as your SQP run.
# If Tf remains weak, try W_OBJ_TF=30.
W_OBJ_PHI = 10.0
W_OBJ_TS  = 10.0
W_OBJ_TF  = 10.0

# Adam for theta only
LR_THETA = 1e-4

# ALM rho initial values
RHO_PDE_PHI0 = 1.0
RHO_PDE_TS0  = 1.0
RHO_PDE_TF0  = 1.0
RHO_BC_PHI0  = 1.0
RHO_BC_TS0   = 1.0
RHO_BC_TF0   = 1.0
RHO_IFACE0   = 1.0
RHO_IC0      = 1.0

RHO_GROWTH = 1.2
RHO_MAX = 1e2
IMPROVE_THRESHOLD = 0.90

# Do not increase rho once a block is already below its tolerance.
TOL_PDE = 1e-6
TOL_BC  = 1e-6
TOL_IF  = 1e-6
TOL_IC  = 1e-6

# Optional validation dataset
NPZ_PATH = "sqp_dataset.npz"
SAVE_NAME = "theta_alm_coupled_lbfgsb_fixed_no_noise_best.npy"


# ============================================================
# Physics constants
# ============================================================
D_PHI = 0.05
SIGMA_A = 1.0
RHO_S_CP_S = 1.0
RHO_F_CP_F = 1.0
K_S = 0.50
K_F = 0.15
GAMMA = 10.0
VY = 1.0


# ============================================================
# Targets / BCs / ICs
# ============================================================
@jax.jit
def phi_left_source(y, t):
    return (DTYPE(1.0) - jnp.exp(-DTYPE(5.0) * t)) * (
        DTYPE(1.0) + DTYPE(0.2) * jnp.sin(jnp.pi * y) * jnp.sin(jnp.pi * y)
    )


@jax.jit
def phi_ic_target(y):
    return jnp.zeros_like(y)


@jax.jit
def Ts_ic_target(x, y):
    return jnp.zeros_like(x)


@jax.jit
def Tf_ic_target(x, y):
    return jnp.zeros_like(x)


# ============================================================
# Network
# ============================================================
def init_mlp_params(key, layer_sizes):
    params = []
    keys = random.split(key, len(layer_sizes) - 1)
    for k, (m, n) in zip(keys, zip(layer_sizes[:-1], layer_sizes[1:])):
        W = random.normal(k, (m, n), dtype=DTYPE) * jnp.sqrt(DTYPE(2.0) / DTYPE(m))
        b = jnp.zeros((n,), dtype=DTYPE)
        params.append({"W": W, "b": b})
    return params


def normalize_xyt(X):
    x = X[:, 0:1]
    y = X[:, 1:2]
    t = X[:, 2:3]

    x_n = DTYPE(2.0) * (x - x_min) / (x_max - x_min + EPS) - DTYPE(1.0)
    y_n = DTYPE(2.0) * (y - y_min) / (y_max - y_min + EPS) - DTYPE(1.0)
    t_n = DTYPE(2.0) * (t - t_min) / (t_max - t_min + EPS) - DTYPE(1.0)

    return jnp.concatenate([x_n, y_n, t_n], axis=1)


def mlp_apply(params, X):
    h = normalize_xyt(X)
    for i, layer in enumerate(params):
        h = h @ layer["W"] + layer["b"]
        if i < len(params) - 1:
            h = jnp.tanh(h)
    return h


def flatten_params(params):
    flat_parts = []
    shapes = []
    for layer in params:
        W, b = layer["W"], layer["b"]
        flat_parts.append(W.reshape(-1))
        flat_parts.append(b.reshape(-1))
        shapes.append((W.shape, b.shape))
    return jnp.concatenate(flat_parts).astype(DTYPE), tuple(shapes)


def unflatten_params(theta, shapes):
    params = []
    idx = 0
    for W_shape, b_shape in shapes:
        W_size = math.prod(W_shape)
        b_size = math.prod(b_shape)

        W = theta[idx:idx + W_size].reshape(W_shape)
        idx += W_size

        b = theta[idx:idx + b_size].reshape(b_shape)
        idx += b_size

        params.append({"W": W, "b": b})

    if idx != theta.size:
        raise ValueError(f"Used {idx}, theta size {theta.size}")

    return params


# ============================================================
# Utilities
# ============================================================
def segment_sum(values, segment_ids, num_segments):
    out = jnp.zeros((num_segments,), dtype=values.dtype)
    return out.at[segment_ids].add(values)


def _rms_np(a):
    a = np.asarray(a, dtype=np.float64)
    return float(np.sqrt(np.mean(a * a) + 1e-30))


def _maxabs_np(a):
    a = np.asarray(a, dtype=np.float64)
    return float(np.max(np.abs(a))) if a.size else 0.0


# ============================================================
# Fixed stratified sampling
# ============================================================
def sample_stratified_3d(key, x0, x1, y0, y1, t0, t1, NX, NY, NT):
    dx = (DTYPE(x1) - DTYPE(x0)) / DTYPE(NX)
    dy = (DTYPE(y1) - DTYPE(y0)) / DTYPE(NY)
    dt = (DTYPE(t1) - DTYPE(t0)) / DTYPE(NT)

    it, iy, ix = jnp.meshgrid(
        jnp.arange(NT),
        jnp.arange(NY),
        jnp.arange(NX),
        indexing="ij",
    )

    it = it.reshape(-1).astype(jnp.int32)
    iy = iy.reshape(-1).astype(jnp.int32)
    ix = ix.reshape(-1).astype(jnp.int32)

    ids = (it * (NY * NX) + iy * NX + ix).astype(jnp.int32)

    xb = DTYPE(x0) + DTYPE(ix) * dx
    yb = DTYPE(y0) + DTYPE(iy) * dy
    tb = DTYPE(t0) + DTYPE(it) * dt

    K = NX * NY * NT
    u = random.uniform(key, (K, 3), minval=0.0, maxval=1.0, dtype=DTYPE)

    xs = xb + u[:, 0] * dx
    ys = yb + u[:, 1] * dy
    ts = tb + u[:, 2] * dt

    X = jnp.stack([xs, ys, ts], axis=1)
    return X, ids


def sample_bc_time_binned(
    key,
    K,
    *,
    x_fixed=None,
    y_fixed=None,
    t0=t_min,
    t1=t_max,
    x_lo=None,
    x_hi=None,
    y_lo=None,
    y_hi=None,
):
    dt = (DTYPE(t1) - DTYPE(t0)) / DTYPE(K)
    j = jnp.arange(K, dtype=jnp.int32)
    tb = DTYPE(t0) + DTYPE(j) * dt

    key, kt, kr = random.split(key, 3)
    u_t = random.uniform(kt, (K, N_BC_PER_BIN), minval=0.0, maxval=1.0, dtype=DTYPE)
    ts = (tb[:, None] + u_t * dt).reshape(-1, 1)
    ids = jnp.repeat(j, N_BC_PER_BIN)

    if x_fixed is not None:
        lo = DTYPE(y_min if y_lo is None else y_lo)
        hi = DTYPE(y_max if y_hi is None else y_hi)
        y = random.uniform(kr, (ts.shape[0], 1), minval=lo, maxval=hi, dtype=DTYPE)
        x = DTYPE(x_fixed) * jnp.ones_like(y)
        return jnp.concatenate([x, y, ts], axis=1), ids

    if y_fixed is not None:
        lo = DTYPE(x_min if x_lo is None else x_lo)
        hi = DTYPE(x_max if x_hi is None else x_hi)
        x = random.uniform(kr, (ts.shape[0], 1), minval=lo, maxval=hi, dtype=DTYPE)
        y = DTYPE(y_fixed) * jnp.ones_like(x)
        return jnp.concatenate([x, y, ts], axis=1), ids

    raise ValueError("Provide x_fixed or y_fixed")


def sample_ic_xy(key, x0, x1, y0, y1, NX, NY):
    dx = (DTYPE(x1) - DTYPE(x0)) / DTYPE(NX)
    dy = (DTYPE(y1) - DTYPE(y0)) / DTYPE(NY)

    iy, ix = jnp.meshgrid(jnp.arange(NY), jnp.arange(NX), indexing="ij")
    iy = iy.reshape(-1).astype(jnp.int32)
    ix = ix.reshape(-1).astype(jnp.int32)

    ids0 = (iy * NX + ix).astype(jnp.int32)

    xb = DTYPE(x0) + DTYPE(ix) * dx
    yb = DTYPE(y0) + DTYPE(iy) * dy

    u = random.uniform(key, (NX * NY, N_IC_PER_BIN, 2), minval=0.0, maxval=1.0, dtype=DTYPE)

    xs = (xb[:, None] + u[:, :, 0] * dx).reshape(-1, 1)
    ys = (yb[:, None] + u[:, :, 1] * dy).reshape(-1, 1)
    ts = t_min * jnp.ones_like(xs)

    X = jnp.concatenate([xs, ys, ts], axis=1)
    ids = jnp.repeat(ids0, N_IC_PER_BIN)

    return X, ids


def sample_interface_yt(key):
    dy = (y_max - y_min) / DTYPE(NY_IF)
    dt = (t_max - t_min) / DTYPE(NT_IF)

    it, iy = jnp.meshgrid(jnp.arange(NT_IF), jnp.arange(NY_IF), indexing="ij")
    it = it.reshape(-1).astype(jnp.int32)
    iy = iy.reshape(-1).astype(jnp.int32)

    ids0 = (it * NY_IF + iy).astype(jnp.int32)

    yb = y_min + DTYPE(iy) * dy
    tb = t_min + DTYPE(it) * dt

    u = random.uniform(key, (NY_IF * NT_IF, N_IF_PER_BIN, 2), minval=0.0, maxval=1.0, dtype=DTYPE)

    ys = (yb[:, None] + u[:, :, 0] * dy).reshape(-1, 1)
    ts = (tb[:, None] + u[:, :, 1] * dt).reshape(-1, 1)
    xs = DTYPE(Ls) * jnp.ones_like(ys)

    X = jnp.concatenate([xs, ys, ts], axis=1)
    ids = jnp.repeat(ids0, N_IF_PER_BIN)

    return X, ids


# ============================================================
# PDE residuals
# ============================================================
def forward3(params, xyt):
    return mlp_apply(params, xyt[None, :])[0, :]


def eval_fields_and_derivs(params, X):
    out = vmap(lambda z: forward3(params, z))(X)
    J = vmap(jacrev(lambda z: forward3(params, z)))(X)
    H = vmap(hessian(lambda z: forward3(params, z)))(X)
    return out, J, H


def residual_phi(params, X, D_phi, Sigma_a):
    out, J, H = eval_fields_and_derivs(params, X)
    phi = out[:, 0]
    phi_t = J[:, 0, 2]
    phi_xx = H[:, 0, 0, 0]
    phi_yy = H[:, 0, 1, 1]
    return phi_t - DTYPE(D_phi) * (phi_xx + phi_yy) + DTYPE(Sigma_a) * phi


def residual_Ts(params, X, rho_s_cp_s, k_s, gamma):
    out, J, H = eval_fields_and_derivs(params, X)
    phi = out[:, 0]
    Ts_t = J[:, 1, 2]
    Ts_xx = H[:, 1, 0, 0]
    Ts_yy = H[:, 1, 1, 1]
    return DTYPE(rho_s_cp_s) * Ts_t - DTYPE(k_s) * (Ts_xx + Ts_yy) - DTYPE(gamma) * phi


def residual_Tf(params, X, rho_f_cp_f, k_f, vy):
    out, J, H = eval_fields_and_derivs(params, X)
    Tf_t = J[:, 2, 2]
    Tf_y = J[:, 2, 1]
    Tf_xx = H[:, 2, 0, 0]
    Tf_yy = H[:, 2, 1, 1]
    return DTYPE(rho_f_cp_f) * Tf_t + DTYPE(vy) * Tf_y - DTYPE(k_f) * (Tf_xx + Tf_yy)


# ============================================================
# Objective and constraints
# ============================================================
@partial(jax.jit, static_argnames=("shapes",))
def objective_value(theta, shapes,
                    X_obj_phi, X_obj_Ts, X_obj_Tf,
                    w_obj_phi, w_obj_Ts, w_obj_Tf,
                    D_phi, Sigma_a, rho_s_cp_s, k_s, gamma,
                    rho_f_cp_f, k_f, vy):
    params = unflatten_params(theta, shapes)

    r_phi = residual_phi(params, X_obj_phi, D_phi, Sigma_a)
    r_Ts = residual_Ts(params, X_obj_Ts, rho_s_cp_s, k_s, gamma)
    r_Tf = residual_Tf(params, X_obj_Tf, rho_f_cp_f, k_f, vy)

    return (
        DTYPE(w_obj_phi) * jnp.mean(r_phi ** 2)
        + DTYPE(w_obj_Ts) * jnp.mean(r_Ts ** 2)
        + DTYPE(w_obj_Tf) * jnp.mean(r_Tf ** 2)
    )


@partial(jax.jit, static_argnames=("shapes",))
def objective_parts(theta, shapes,
                    X_obj_phi, X_obj_Ts, X_obj_Tf,
                    D_phi, Sigma_a, rho_s_cp_s, k_s, gamma,
                    rho_f_cp_f, k_f, vy):
    params = unflatten_params(theta, shapes)

    r_phi = residual_phi(params, X_obj_phi, D_phi, Sigma_a)
    r_Ts = residual_Ts(params, X_obj_Ts, rho_s_cp_s, k_s, gamma)
    r_Tf = residual_Tf(params, X_obj_Tf, rho_f_cp_f, k_f, vy)

    return (
        jnp.sqrt(jnp.mean(r_phi ** 2) + EPS),
        jnp.sqrt(jnp.mean(r_Ts ** 2) + EPS),
        jnp.sqrt(jnp.mean(r_Tf ** 2) + EPS),
    )


@partial(jax.jit, static_argnames=("shapes",))
def constraint_vector(theta, shapes,
                      X_con_phi, ids_con_phi,
                      X_con_Ts, ids_con_Ts,
                      X_con_Tf, ids_con_Tf,
                      X_phi_left, ids_phi_left,
                      X_phi_y0, ids_phi_y0,
                      X_phi_y1, ids_phi_y1,
                      X_phi_if, ids_phi_if,
                      X_Ts_x0, ids_Ts_x0,
                      X_Ts_y0, ids_Ts_y0,
                      X_Ts_y1, ids_Ts_y1,
                      X_Tf_y0, ids_Tf_y0,
                      X_Tf_y1, ids_Tf_y1,
                      X_Tf_x1, ids_Tf_x1,
                      X_if, ids_if,
                      X_ic_phi, ids_ic_phi,
                      X_ic_Ts, ids_ic_Ts,
                      X_ic_Tf, ids_ic_Tf,
                      D_phi, Sigma_a, rho_s_cp_s, k_s, gamma,
                      rho_f_cp_f, k_f, vy):
    params = unflatten_params(theta, shapes)

    # PDE anchors
    r_phi = residual_phi(params, X_con_phi, D_phi, Sigma_a)
    r_Ts = residual_Ts(params, X_con_Ts, rho_s_cp_s, k_s, gamma)
    r_Tf = residual_Tf(params, X_con_Tf, rho_f_cp_f, k_f, vy)

    c_pde_phi = segment_sum(r_phi, ids_con_phi, K_CON_PHI) / DTYPE(N_PER_CELL)
    c_pde_Ts = segment_sum(r_Ts, ids_con_Ts, K_CON_TS) / DTYPE(N_PER_CELL)
    c_pde_Tf = segment_sum(r_Tf, ids_con_Tf, K_CON_TF) / DTYPE(N_PER_CELL)

    def phi_fun(z):
        return forward3(params, z)[0]

    def Ts_fun(z):
        return forward3(params, z)[1]

    def Tf_fun(z):
        return forward3(params, z)[2]

    # Neutron BCs
    phiL = mlp_apply(params, X_phi_left)[:, 0]
    yL, tL = X_phi_left[:, 1], X_phi_left[:, 2]
    c_phi_left = segment_sum(phiL - phi_left_source(yL, tL), ids_phi_left, K_BC_PHI_LEFT) / DTYPE(N_BC_PER_BIN)

    phi_y0 = vmap(lambda z: jacrev(phi_fun)(z)[1])(X_phi_y0)
    c_phi_y0 = segment_sum(phi_y0, ids_phi_y0, K_BC_PHI_Y0) / DTYPE(N_BC_PER_BIN)

    phi_y1 = vmap(lambda z: jacrev(phi_fun)(z)[1])(X_phi_y1)
    c_phi_y1 = segment_sum(phi_y1, ids_phi_y1, K_BC_PHI_Y1) / DTYPE(N_BC_PER_BIN)

    phi_x_if = vmap(lambda z: jacrev(phi_fun)(z)[0])(X_phi_if)
    c_phi_if = segment_sum(phi_x_if, ids_phi_if, K_BC_PHI_IF) / DTYPE(N_BC_PER_BIN)

    c_bc_phi = jnp.concatenate([c_phi_left, c_phi_y0, c_phi_y1, c_phi_if], axis=0)

    # Fuel thermal BCs
    Ts_x0 = vmap(lambda z: jacrev(Ts_fun)(z)[0])(X_Ts_x0)
    c_Ts_x0 = segment_sum(Ts_x0, ids_Ts_x0, K_BC_TS_X0) / DTYPE(N_BC_PER_BIN)

    Ts_y0 = vmap(lambda z: jacrev(Ts_fun)(z)[1])(X_Ts_y0)
    c_Ts_y0 = segment_sum(Ts_y0, ids_Ts_y0, K_BC_TS_Y0) / DTYPE(N_BC_PER_BIN)

    Ts_y1 = vmap(lambda z: jacrev(Ts_fun)(z)[1])(X_Ts_y1)
    c_Ts_y1 = segment_sum(Ts_y1, ids_Ts_y1, K_BC_TS_Y1) / DTYPE(N_BC_PER_BIN)

    c_bc_Ts = jnp.concatenate([c_Ts_x0, c_Ts_y0, c_Ts_y1], axis=0)

    # Coolant thermal BCs
    Tf_bottom = mlp_apply(params, X_Tf_y0)[:, 2]
    c_Tf_y0 = segment_sum(Tf_bottom, ids_Tf_y0, K_BC_TF_Y0) / DTYPE(N_BC_PER_BIN)

    Tf_y1 = vmap(lambda z: jacrev(Tf_fun)(z)[1])(X_Tf_y1)
    c_Tf_y1 = segment_sum(Tf_y1, ids_Tf_y1, K_BC_TF_Y1) / DTYPE(N_BC_PER_BIN)

    Tf_x1 = vmap(lambda z: jacrev(Tf_fun)(z)[0])(X_Tf_x1)
    c_Tf_x1 = segment_sum(Tf_x1, ids_Tf_x1, K_BC_TF_X1) / DTYPE(N_BC_PER_BIN)

    c_bc_Tf = jnp.concatenate([c_Tf_y0, c_Tf_y1, c_Tf_x1], axis=0)

    # Interface
    out_if = mlp_apply(params, X_if)
    Ts_if = out_if[:, 1]
    Tf_if = out_if[:, 2]

    c_if_T = segment_sum(Ts_if - Tf_if, ids_if, K_IF) / DTYPE(N_IF_PER_BIN)

    Tsx_if = vmap(lambda z: jacrev(Ts_fun)(z)[0])(X_if)
    Tfx_if = vmap(lambda z: jacrev(Tf_fun)(z)[0])(X_if)

    flux_jump = -DTYPE(k_s) * Tsx_if + DTYPE(k_f) * Tfx_if
    c_if_q = segment_sum(flux_jump, ids_if, K_IF) / DTYPE(N_IF_PER_BIN)

    c_if = jnp.concatenate([c_if_T, c_if_q], axis=0)

    # ICs
    out_phi0 = mlp_apply(params, X_ic_phi)
    out_Ts0 = mlp_apply(params, X_ic_Ts)
    out_Tf0 = mlp_apply(params, X_ic_Tf)

    phi0 = out_phi0[:, 0]
    Ts0 = out_Ts0[:, 1]
    Tf0 = out_Tf0[:, 2]

    y_phi0 = X_ic_phi[:, 1]
    c_ic_phi = segment_sum(phi0 - phi_ic_target(y_phi0), ids_ic_phi, K_IC_PHI) / DTYPE(N_IC_PER_BIN)
    c_ic_Ts = segment_sum(Ts0 - Ts_ic_target(X_ic_Ts[:, 0], X_ic_Ts[:, 1]), ids_ic_Ts, K_IC_TS) / DTYPE(N_IC_PER_BIN)
    c_ic_Tf = segment_sum(Tf0 - Tf_ic_target(X_ic_Tf[:, 0], X_ic_Tf[:, 1]), ids_ic_Tf, K_IC_TF) / DTYPE(N_IC_PER_BIN)

    c_ic = jnp.concatenate([c_ic_phi, c_ic_Ts, c_ic_Tf], axis=0)

    return jnp.concatenate([
        c_pde_phi,
        c_pde_Ts,
        c_pde_Tf,
        c_bc_phi,
        c_bc_Ts,
        c_bc_Tf,
        c_if,
        c_ic,
    ], axis=0)


def block_alm(c, lam, sl, rho):
    cb = c[sl]
    lb = lam[sl]
    return jnp.mean(lb * cb) + DTYPE(0.5) * DTYPE(rho) * jnp.mean(cb ** 2)


@partial(jax.jit, static_argnames=("shapes",))
def alm_loss_and_grad(theta, shapes, lam,
                      rho_pde_phi, rho_pde_Ts, rho_pde_Tf,
                      rho_bc_phi, rho_bc_Ts, rho_bc_Tf,
                      rho_iface, rho_ic,
                      X_obj_phi, X_obj_Ts, X_obj_Tf,
                      X_con_phi, ids_con_phi,
                      X_con_Ts, ids_con_Ts,
                      X_con_Tf, ids_con_Tf,
                      X_phi_left, ids_phi_left,
                      X_phi_y0, ids_phi_y0,
                      X_phi_y1, ids_phi_y1,
                      X_phi_if, ids_phi_if,
                      X_Ts_x0, ids_Ts_x0,
                      X_Ts_y0, ids_Ts_y0,
                      X_Ts_y1, ids_Ts_y1,
                      X_Tf_y0, ids_Tf_y0,
                      X_Tf_y1, ids_Tf_y1,
                      X_Tf_x1, ids_Tf_x1,
                      X_if, ids_if,
                      X_ic_phi, ids_ic_phi,
                      X_ic_Ts, ids_ic_Ts,
                      X_ic_Tf, ids_ic_Tf,
                      w_obj_phi, w_obj_Ts, w_obj_Tf,
                      D_phi, Sigma_a, rho_s_cp_s, k_s, gamma,
                      rho_f_cp_f, k_f, vy):
    def loss_fn(th):
        obj = objective_value(
            th, shapes,
            X_obj_phi, X_obj_Ts, X_obj_Tf,
            w_obj_phi, w_obj_Ts, w_obj_Tf,
            D_phi, Sigma_a, rho_s_cp_s, k_s, gamma,
            rho_f_cp_f, k_f, vy,
        )

        c = constraint_vector(
            th, shapes,
            X_con_phi, ids_con_phi,
            X_con_Ts, ids_con_Ts,
            X_con_Tf, ids_con_Tf,
            X_phi_left, ids_phi_left,
            X_phi_y0, ids_phi_y0,
            X_phi_y1, ids_phi_y1,
            X_phi_if, ids_phi_if,
            X_Ts_x0, ids_Ts_x0,
            X_Ts_y0, ids_Ts_y0,
            X_Ts_y1, ids_Ts_y1,
            X_Tf_y0, ids_Tf_y0,
            X_Tf_y1, ids_Tf_y1,
            X_Tf_x1, ids_Tf_x1,
            X_if, ids_if,
            X_ic_phi, ids_ic_phi,
            X_ic_Ts, ids_ic_Ts,
            X_ic_Tf, ids_ic_Tf,
            D_phi, Sigma_a, rho_s_cp_s, k_s, gamma,
            rho_f_cp_f, k_f, vy,
        )

        alm = (
            block_alm(c, lam, BLOCKS["pde_phi"], rho_pde_phi)
            + block_alm(c, lam, BLOCKS["pde_Ts"], rho_pde_Ts)
            + block_alm(c, lam, BLOCKS["pde_Tf"], rho_pde_Tf)
            + block_alm(c, lam, BLOCKS["bc_phi"], rho_bc_phi)
            + block_alm(c, lam, BLOCKS["bc_Ts"], rho_bc_Ts)
            + block_alm(c, lam, BLOCKS["bc_Tf"], rho_bc_Tf)
            + block_alm(c, lam, BLOCKS["iface"], rho_iface)
            + block_alm(c, lam, BLOCKS["ic"], rho_ic)
        )

        loss = obj + alm

        aux = {
            "obj": obj,
            "alm": alm,
            "feas": jnp.sqrt(jnp.mean(c ** 2) + EPS),
            "feas_max": jnp.max(jnp.abs(c)),
        }
        return loss, aux

    (loss, aux), g = jax.value_and_grad(loss_fn, has_aux=True)(theta)
    return loss, g, aux


@partial(jax.jit, static_argnames=("shapes",))
def constraints_only(theta, shapes,
                     X_con_phi, ids_con_phi,
                     X_con_Ts, ids_con_Ts,
                     X_con_Tf, ids_con_Tf,
                     X_phi_left, ids_phi_left,
                     X_phi_y0, ids_phi_y0,
                     X_phi_y1, ids_phi_y1,
                     X_phi_if, ids_phi_if,
                     X_Ts_x0, ids_Ts_x0,
                     X_Ts_y0, ids_Ts_y0,
                     X_Ts_y1, ids_Ts_y1,
                     X_Tf_y0, ids_Tf_y0,
                     X_Tf_y1, ids_Tf_y1,
                     X_Tf_x1, ids_Tf_x1,
                     X_if, ids_if,
                     X_ic_phi, ids_ic_phi,
                     X_ic_Ts, ids_ic_Ts,
                     X_ic_Tf, ids_ic_Tf,
                     D_phi, Sigma_a, rho_s_cp_s, k_s, gamma,
                     rho_f_cp_f, k_f, vy):
    return constraint_vector(
        theta, shapes,
        X_con_phi, ids_con_phi,
        X_con_Ts, ids_con_Ts,
        X_con_Tf, ids_con_Tf,
        X_phi_left, ids_phi_left,
        X_phi_y0, ids_phi_y0,
        X_phi_y1, ids_phi_y1,
        X_phi_if, ids_phi_if,
        X_Ts_x0, ids_Ts_x0,
        X_Ts_y0, ids_Ts_y0,
        X_Ts_y1, ids_Ts_y1,
        X_Tf_y0, ids_Tf_y0,
        X_Tf_y1, ids_Tf_y1,
        X_Tf_x1, ids_Tf_x1,
        X_if, ids_if,
        X_ic_phi, ids_ic_phi,
        X_ic_Ts, ids_ic_Ts,
        X_ic_Tf, ids_ic_Tf,
        D_phi, Sigma_a, rho_s_cp_s, k_s, gamma,
        rho_f_cp_f, k_f, vy,
    )


# ============================================================
# Adam
# ============================================================
@jax.jit
def adam_step(theta, g, m, v, step, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    m = beta1 * m + (1.0 - beta1) * g
    v = beta2 * v + (1.0 - beta2) * (g * g)

    mhat = m / (1.0 - beta1 ** step)
    vhat = v / (1.0 - beta2 ** step)

    theta = theta - DTYPE(lr) * mhat / (jnp.sqrt(vhat) + eps)

    return theta, m, v


# ============================================================
# Validation against sqp_dataset.npz
# ============================================================
@partial(jax.jit, static_argnames=("shapes",))
def predict_batch(theta, shapes, X):
    params = unflatten_params(theta, shapes)
    return mlp_apply(params, X)


def rel_l2_num_den(pred, true):
    pred = np.asarray(pred, dtype=np.float64)
    true = np.asarray(true, dtype=np.float64)
    num = np.sum((pred - true) ** 2)
    den = np.sum(true ** 2) + 1e-30
    return num, den


def validate_against_npz(theta, shapes, npz_path=NPZ_PATH, batch_size=50000):
    path = Path(npz_path)
    if not path.exists():
        return None

    data = np.load(path, allow_pickle=True)

    x = np.asarray(data["x"], dtype=np.float64)
    y = np.asarray(data["y"], dtype=np.float64)
    tt = np.asarray(data["t"] if "t" in data else data["time"], dtype=np.float64)

    T_ref = np.asarray(data["T"], dtype=np.float64)
    phi_ref = np.asarray(data["phi"], dtype=np.float64)

    nt, nx, ny = T_ref.shape

    fuel_mask = x < float(Ls) - 1e-12
    cool_mask = x > float(Ls) + 1e-12

    XX, YY = np.meshgrid(x, y, indexing="ij")

    num_phi = den_phi = 0.0
    num_Ts = den_Ts = 0.0
    num_Tf = den_Tf = 0.0

    for k, tk in enumerate(tt):
        TT = np.full_like(XX, float(tk), dtype=np.float64)
        X_eval = np.stack([XX.reshape(-1), YY.reshape(-1), TT.reshape(-1)], axis=1)

        preds = []
        for i in range(0, X_eval.shape[0], batch_size):
            Xb = jnp.asarray(X_eval[i:i + batch_size], dtype=DTYPE)
            preds.append(np.asarray(predict_batch(theta, shapes, Xb)))
        pred = np.vstack(preds).reshape(nx, ny, 3)

        phi_p = pred[:, :, 0]
        Ts_p = pred[:, :, 1]
        Tf_p = pred[:, :, 2]

        n, d = rel_l2_num_den(phi_p[fuel_mask, :], phi_ref[k, fuel_mask, :])
        num_phi += n; den_phi += d

        n, d = rel_l2_num_den(Ts_p[fuel_mask, :], T_ref[k, fuel_mask, :])
        num_Ts += n; den_Ts += d

        n, d = rel_l2_num_den(Tf_p[cool_mask, :], T_ref[k, cool_mask, :])
        num_Tf += n; den_Tf += d

    return {
        "phi_rel": float(np.sqrt(num_phi / den_phi)),
        "Ts_rel": float(np.sqrt(num_Ts / den_Ts)),
        "Tf_rel": float(np.sqrt(num_Tf / den_Tf)),
    }


# ============================================================
# Fixed base sampling for ALM
# ============================================================
def build_fixed_sets(key):
    # Same order as your SQP code after network initialization.
    key, k_obj_phi, k_obj_Ts, k_obj_Tf = random.split(key, 4)

    X_obj_phi, _ = sample_stratified_3d(k_obj_phi, x_min, Ls, y_min, y_max, t_min, t_max,
                                        NX_OBJ_PHI, NY_OBJ_PHI, NT_OBJ_PHI)
    X_obj_Ts, _ = sample_stratified_3d(k_obj_Ts, x_min, Ls, y_min, y_max, t_min, t_max,
                                       NX_OBJ_TS, NY_OBJ_TS, NT_OBJ_TS)
    X_obj_Tf, _ = sample_stratified_3d(k_obj_Tf, Ls, x_max, y_min, y_max, t_min, t_max,
                                       NX_OBJ_TF, NY_OBJ_TF, NT_OBJ_TF)

    key, k_con_phi, k_con_Ts, k_con_Tf = random.split(key, 4)

    X_con_phi, ids_con_phi = sample_stratified_3d(k_con_phi, x_min, Ls, y_min, y_max, t_min, t_max,
                                                  NX_CON_PHI, NY_CON_PHI, NT_CON_PHI)
    X_con_Ts, ids_con_Ts = sample_stratified_3d(k_con_Ts, x_min, Ls, y_min, y_max, t_min, t_max,
                                                NX_CON_TS, NY_CON_TS, NT_CON_TS)
    X_con_Tf, ids_con_Tf = sample_stratified_3d(k_con_Tf, Ls, x_max, y_min, y_max, t_min, t_max,
                                                NX_CON_TF, NY_CON_TF, NT_CON_TF)

    key, k1, k2, k3, k4, k5, k6, k7, k8, k9, k10, k11, k12, k13 = random.split(key, 14)

    X_phi_left, ids_phi_left = sample_bc_time_binned(k1, K_BC_PHI_LEFT, x_fixed=x_min, y_lo=y_min, y_hi=y_max)
    X_phi_y0, ids_phi_y0 = sample_bc_time_binned(k2, K_BC_PHI_Y0, y_fixed=y_min, x_lo=x_min, x_hi=Ls)
    X_phi_y1, ids_phi_y1 = sample_bc_time_binned(k3, K_BC_PHI_Y1, y_fixed=y_max, x_lo=x_min, x_hi=Ls)
    X_phi_if, ids_phi_if = sample_bc_time_binned(k4, K_BC_PHI_IF, x_fixed=Ls, y_lo=y_min, y_hi=y_max)

    X_Ts_x0, ids_Ts_x0 = sample_bc_time_binned(k5, K_BC_TS_X0, x_fixed=x_min, y_lo=y_min, y_hi=y_max)
    X_Ts_y0, ids_Ts_y0 = sample_bc_time_binned(k6, K_BC_TS_Y0, y_fixed=y_min, x_lo=x_min, x_hi=Ls)
    X_Ts_y1, ids_Ts_y1 = sample_bc_time_binned(k7, K_BC_TS_Y1, y_fixed=y_max, x_lo=x_min, x_hi=Ls)

    xL = float(DTYPE(Ls) + DTYPE(1e-8))
    xR = float(x_max - DTYPE(1e-8))

    X_Tf_y0, ids_Tf_y0 = sample_bc_time_binned(k8, K_BC_TF_Y0, y_fixed=y_min, x_lo=xL, x_hi=xR)
    X_Tf_y1, ids_Tf_y1 = sample_bc_time_binned(k9, K_BC_TF_Y1, y_fixed=y_max, x_lo=xL, x_hi=xR)
    X_Tf_x1, ids_Tf_x1 = sample_bc_time_binned(k10, K_BC_TF_X1, x_fixed=x_max, y_lo=y_min, y_hi=y_max)

    X_if, ids_if = sample_interface_yt(k11)

    X_ic_phi, ids_ic_phi = sample_ic_xy(k12, x_min, Ls, y_min, y_max, NX_IC, NY_IC)
    X_ic_Ts, ids_ic_Ts = sample_ic_xy(k13, x_min, Ls, y_min, y_max, NX_IC, NY_IC)

    key, k_ic_tf = random.split(key)
    X_ic_Tf, ids_ic_Tf = sample_ic_xy(k_ic_tf, Ls, x_max, y_min, y_max, NX_IC, NY_IC)

    obj_args = (X_obj_phi, X_obj_Ts, X_obj_Tf)

    con_args = (
        X_con_phi, ids_con_phi,
        X_con_Ts, ids_con_Ts,
        X_con_Tf, ids_con_Tf,
        X_phi_left, ids_phi_left,
        X_phi_y0, ids_phi_y0,
        X_phi_y1, ids_phi_y1,
        X_phi_if, ids_phi_if,
        X_Ts_x0, ids_Ts_x0,
        X_Ts_y0, ids_Ts_y0,
        X_Ts_y1, ids_Ts_y1,
        X_Tf_y0, ids_Tf_y0,
        X_Tf_y1, ids_Tf_y1,
        X_Tf_x1, ids_Tf_x1,
        X_if, ids_if,
        X_ic_phi, ids_ic_phi,
        X_ic_Ts, ids_ic_Ts,
        X_ic_Tf, ids_ic_Tf,
    )

    return obj_args, con_args


# ============================================================
# Rho update helpers
# ============================================================
def rho_update(name, rho, cur, prev):
    tol = TOL_PDE if name.startswith("pde") else TOL_IF if name == "iface" else TOL_IC if name == "ic" else TOL_BC
    if prev is None:
        return rho
    if cur <= tol:
        return rho
    if cur > IMPROVE_THRESHOLD * prev:
        return min(RHO_GROWTH * rho, RHO_MAX)
    return rho


def update_lambda_block(lam, c, name, rho):
    sl = BLOCKS[name]
    return lam.at[sl].set(lam[sl] + DTYPE(rho) * c[sl])


def print_constraint_blocks(c_np, prefix="blocks"):
    print(prefix)
    for name in ["pde_phi", "pde_Ts", "pde_Tf", "bc_phi", "bc_Ts", "bc_Tf", "iface", "ic"]:
        v = c_np[BLOCKS[name]]
        print(f"  {name:10s}: rms={_rms_np(v):.3e}  max={_maxabs_np(v):.3e}")


# ============================================================
# Training
# ============================================================
def make_lbfgs_objective(theta_dtype_template, shapes, lam, rhos, obj_args, con_args):
    X_obj_phi, X_obj_Ts, X_obj_Tf = obj_args

    def fun_and_jac(theta_np):
        th = jnp.asarray(theta_np, dtype=DTYPE)
        loss, g, aux = alm_loss_and_grad(
            th, shapes, lam,
            DTYPE(rhos["pde_phi"]), DTYPE(rhos["pde_Ts"]), DTYPE(rhos["pde_Tf"]),
            DTYPE(rhos["bc_phi"]), DTYPE(rhos["bc_Ts"]), DTYPE(rhos["bc_Tf"]),
            DTYPE(rhos["iface"]), DTYPE(rhos["ic"]),
            X_obj_phi, X_obj_Ts, X_obj_Tf,
            *con_args,
            DTYPE(W_OBJ_PHI), DTYPE(W_OBJ_TS), DTYPE(W_OBJ_TF),
            DTYPE(D_PHI), DTYPE(SIGMA_A), DTYPE(RHO_S_CP_S), DTYPE(K_S), DTYPE(GAMMA),
            DTYPE(RHO_F_CP_F), DTYPE(K_F), DTYPE(VY),
        )
        return float(loss), np.asarray(g, dtype=np.float64)

    return fun_and_jac


def train_alm_lbfgsb(
    seed=0,
    hidden_dim=30,
    num_hidden=3,
    total_lbfgs_budget=TOTAL_LBFGS_BUDGET,
    inner_maxiter=INNER_MAXITER,
    validate_every_outer=VALIDATE_EVERY_OUTER,
):
    key = random.PRNGKey(seed)

    # Network initialization follows your SQP code.
    layer_sizes = [3] + [hidden_dim] * num_hidden + [3]
    key, k0 = random.split(key)
    params0 = init_mlp_params(k0, layer_sizes)
    theta, shapes = flatten_params(params0)

    # Fixed base sets: sampled once, then held fixed.
    obj_args, con_args = build_fixed_sets(key)
    X_obj_phi, X_obj_Ts, X_obj_Tf = obj_args

    max_outer = int(math.ceil(total_lbfgs_budget / inner_maxiter))

    lam = jnp.zeros((M_CON,), dtype=DTYPE)

    # Blockwise rho values: same blocks as the Adam-inner neutron ALM.
    rhos = {
        "pde_phi": float(RHO_PDE_PHI0),
        "pde_Ts": float(RHO_PDE_TS0),
        "pde_Tf": float(RHO_PDE_TF0),
        "bc_phi": float(RHO_BC_PHI0),
        "bc_Ts": float(RHO_BC_TS0),
        "bc_Tf": float(RHO_BC_TF0),
        "iface": float(RHO_IFACE0),
        "ic": float(RHO_IC0),
    }

    prev_rms = {k: None for k in rhos.keys()}

    best_score = float("inf")
    best_theta = theta
    total_nit = 0
    total_nfev = 0

    t0 = time.time()

    print("Traditional ALM-PINN for coupled neutron/thermal system")
    print("fixed points, no jitter, no resampling")
    print("L-BFGS-B inner theta optimizer; classical lambda update after each outer loop")
    print("n_params =", int(theta.size))
    print("M_CON =", int(M_CON))
    print("objective points:", int(X_obj_phi.shape[0]), int(X_obj_Ts.shape[0]), int(X_obj_Tf.shape[0]))
    print("TOTAL_LBFGS_BUDGET =", int(total_lbfgs_budget))
    print("INNER_MAXITER =", int(inner_maxiter), "MAX_OUTER =", int(max_outer))
    print("L-BFGS-B: ftol=", LBFGS_FTOL, "gtol=", LBFGS_GTOL, "maxls=", LBFGS_MAXLS, "maxcor=", LBFGS_MAXCOR)
    print("weights:", W_OBJ_PHI, W_OBJ_TS, W_OBJ_TF)
    print("initial rhos:", rhos)
    print("rho_growth=", RHO_GROWTH, "rho_max=", RHO_MAX, "improve_threshold=", IMPROVE_THRESHOLD)
    print("npz validation:", NPZ_PATH if Path(NPZ_PATH).exists() else "not found")

    # Warm-up compile.
    _ = alm_loss_and_grad(
        theta, shapes, lam,
        DTYPE(rhos["pde_phi"]), DTYPE(rhos["pde_Ts"]), DTYPE(rhos["pde_Tf"]),
        DTYPE(rhos["bc_phi"]), DTYPE(rhos["bc_Ts"]), DTYPE(rhos["bc_Tf"]),
        DTYPE(rhos["iface"]), DTYPE(rhos["ic"]),
        X_obj_phi, X_obj_Ts, X_obj_Tf,
        *con_args,
        DTYPE(W_OBJ_PHI), DTYPE(W_OBJ_TS), DTYPE(W_OBJ_TF),
        DTYPE(D_PHI), DTYPE(SIGMA_A), DTYPE(RHO_S_CP_S), DTYPE(K_S), DTYPE(GAMMA),
        DTYPE(RHO_F_CP_F), DTYPE(K_F), DTYPE(VY),
    )

    for outer in range(1, max_outer + 1):
        print(
            f"\n[OUTER {outer}/{max_outer}] "
            f"rho_pde=({rhos['pde_phi']:.2e},{rhos['pde_Ts']:.2e},{rhos['pde_Tf']:.2e}) "
            f"rho_bc=({rhos['bc_phi']:.2e},{rhos['bc_Ts']:.2e},{rhos['bc_Tf']:.2e}) "
            f"rho_if={rhos['iface']:.2e} rho_ic={rhos['ic']:.2e}"
        )

        fun_and_jac = make_lbfgs_objective(theta, shapes, lam, rhos, obj_args, con_args)
        theta0_np = np.asarray(theta, dtype=np.float64)

        res = scipy.optimize.minimize(
            fun_and_jac,
            theta0_np,
            method="L-BFGS-B",
            jac=True,
            options={
                "maxiter": int(inner_maxiter),
                "ftol": LBFGS_FTOL,
                "gtol": LBFGS_GTOL,
                "maxls": LBFGS_MAXLS,
                "maxcor": LBFGS_MAXCOR,
                "disp": False,
            },
        )

        theta = jnp.asarray(res.x, dtype=DTYPE)
        total_nit += int(getattr(res, "nit", 0))
        total_nfev += int(getattr(res, "nfev", 0))

        # Evaluate augmented loss at end of inner solve.
        loss, g_end, aux = alm_loss_and_grad(
            theta, shapes, lam,
            DTYPE(rhos["pde_phi"]), DTYPE(rhos["pde_Ts"]), DTYPE(rhos["pde_Tf"]),
            DTYPE(rhos["bc_phi"]), DTYPE(rhos["bc_Ts"]), DTYPE(rhos["bc_Tf"]),
            DTYPE(rhos["iface"]), DTYPE(rhos["ic"]),
            X_obj_phi, X_obj_Ts, X_obj_Tf,
            *con_args,
            DTYPE(W_OBJ_PHI), DTYPE(W_OBJ_TS), DTYPE(W_OBJ_TF),
            DTYPE(D_PHI), DTYPE(SIGMA_A), DTYPE(RHO_S_CP_S), DTYPE(K_S), DTYPE(GAMMA),
            DTYPE(RHO_F_CP_F), DTYPE(K_F), DTYPE(VY),
        )

        print(
            f"  [L-BFGS-B] success={bool(res.success)} nit={int(getattr(res, 'nit', 0)):4d} "
            f"nfev={int(getattr(res, 'nfev', 0)):4d} "
            f"loss={float(loss):.3e} obj={float(aux['obj']):.3e} "
            f"alm={float(aux['alm']):.3e} feas={float(aux['feas']):.3e} "
            f"max={float(aux['feas_max']):.3e} | {str(res.message)[:80]}"
        )

        # End inner loop: evaluate constraints on the same fixed points.
        c = constraints_only(
            theta, shapes,
            *con_args,
            DTYPE(D_PHI), DTYPE(SIGMA_A), DTYPE(RHO_S_CP_S), DTYPE(K_S), DTYPE(GAMMA),
            DTYPE(RHO_F_CP_F), DTYPE(K_F), DTYPE(VY),
        )

        c_np = np.asarray(c)

        # Objective residual diagnostics.
        obj_phi_rms, obj_Ts_rms, obj_Tf_rms = objective_parts(
            theta, shapes,
            X_obj_phi, X_obj_Ts, X_obj_Tf,
            DTYPE(D_PHI), DTYPE(SIGMA_A), DTYPE(RHO_S_CP_S), DTYPE(K_S), DTYPE(GAMMA),
            DTYPE(RHO_F_CP_F), DTYPE(K_F), DTYPE(VY),
        )

        print(
            f"[ALM UPDATE] outer={outer} total_nit={total_nit} total_nfev={total_nfev} "
            f"obj_rms(phi,Ts,Tf)=({float(obj_phi_rms):.3e},{float(obj_Ts_rms):.3e},{float(obj_Tf_rms):.3e})"
        )
        print_constraint_blocks(c_np, "constraint blocks")

        # Classical multiplier update: lambda <- lambda + rho*c.
        for name in rhos.keys():
            lam = update_lambda_block(lam, c, name, rhos[name])

        # Rho update: blockwise.
        cur_rms = {}
        for name in rhos.keys():
            cur_rms[name] = _rms_np(c_np[BLOCKS[name]])
            rhos[name] = rho_update(name, rhos[name], cur_rms[name], prev_rms[name])
            prev_rms[name] = cur_rms[name]

        # Validation and best-save.
        if validate_every_outer and outer % int(validate_every_outer) == 0:
            val = validate_against_npz(theta, shapes, NPZ_PATH)
            if val is not None:
                print(
                    f"[VALID] phi={val['phi_rel']:.3e} "
                    f"Ts={val['Ts_rel']:.3e} Tf={val['Tf_rel']:.3e}"
                )

                score = val["phi_rel"] + val["Ts_rel"] + val["Tf_rel"]
                if score < best_score:
                    best_score = score
                    best_theta = theta
                    np.save(SAVE_NAME, np.asarray(best_theta))
                    print(f"      >>> new best score={best_score:.3e}; saved -> {SAVE_NAME}")
            else:
                score = float(np.sqrt(np.mean(c_np ** 2)))
                if score < best_score:
                    best_score = score
                    best_theta = theta
                    np.save(SAVE_NAME, np.asarray(best_theta))
                    print(f"      >>> new best feasibility={best_score:.3e}; saved -> {SAVE_NAME}")

    elapsed = time.time() - t0
    print(f"\n[done] elapsed={elapsed:.2f}s")

    np.save("theta_alm_coupled_lbfgsb_fixed_no_noise_final.npy", np.asarray(theta))
    print("Saved final theta -> theta_alm_coupled_lbfgsb_fixed_no_noise_final.npy")

    if Path(SAVE_NAME).exists():
        print("Best theta already saved ->", SAVE_NAME)
    else:
        np.save(SAVE_NAME, np.asarray(theta))
        print("Saved best theta ->", SAVE_NAME)

    val = validate_against_npz(theta, shapes, NPZ_PATH)
    if val is not None:
        print(
            f"[FINAL VALID] phi={val['phi_rel']:.6e} "
            f"Ts={val['Ts_rel']:.6e} Tf={val['Tf_rel']:.6e}"
        )

    return theta, shapes, lam

def main():
    theta, shapes, lam = train_alm_lbfgsb(
        seed=0,
        hidden_dim=30,
        num_hidden=3,
        total_lbfgs_budget=TOTAL_LBFGS_BUDGET,
        inner_maxiter=INNER_MAXITER,
        validate_every_outer=VALIDATE_EVERY_OUTER,
    )
    return theta, shapes, lam


if __name__ == "__main__":
    main()


Traditional ALM-PINN for coupled neutron/thermal system
fixed points, no jitter, no resampling
L-BFGS-B inner theta optimizer; classical lambda update after each outer loop
n_params = 2073
M_CON = 2002
objective points: 2250 2250 2250
TOTAL_LBFGS_BUDGET = 50000
INNER_MAXITER = 100 MAX_OUTER = 500
L-BFGS-B: ftol= 1e-12 gtol= 1e-08 maxls= 50 maxcor= 50
weights: 10.0 10.0 10.0
initial rhos: {'pde_phi': 1.0, 'pde_Ts': 1.0, 'pde_Tf': 1.0, 'bc_phi': 1.0, 'bc_Ts': 1.0, 'bc_Tf': 1.0, 'iface': 1.0, 'ic': 1.0}
rho_growth= 1.2 rho_max= 100.0 improve_threshold= 0.9
npz validation: not found

[OUTER 1/500] rho_pde=(1.00e+00,1.00e+00,1.00e+00) rho_bc=(1.00e+00,1.00e+00,1.00e+00) rho_if=1.00e+00 rho_ic=1.00e+00
  [L-BFGS-B] success=False nit= 100 nfev= 102 loss=1.187e-01 obj=9.318e-03 alm=1.093e-01 feas=1.618e-01 max=1.163e+00 | STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT
[ALM UPDATE] outer=1 total_nit=100 total_nfev=102 obj_rms(phi,Ts,Tf)=(1.715e-02,1.633e-02,1.926e-02)
constraint blocks
  pde_phi  

In [0]:
done] elapsed=262.99s
[FINAL] full-grid MSE=2.134e-04, relL2=2.381e-02
[BEST]  full-grid MSE=1.374e-04, relL2=1.911e-02
Saved best theta -> theta_traditional_alm_burgers_lbfg

In [ ]:
full-grid MSE=6.368e-05, relL2=1.301e-02
good type wwhen rell2 = 8e-3